# Predict KL Divergence from Catchment Attributes

## Introduction

We compute the ordinary and linear moment statistics of runoff time series for a large sample of catchments. We then test the predictability of these statistics from catchment attributes using a gradient boosting machine learning approach.  The purpose of testing predictability of the order statistics and L-moments is to fully describe parametric distributions for estimating streamflow distributions (flow duration curves), a common problem in applied hydrology.

Catchment attributes are used as predictors of each order statistic.  Attributes are added cumulatively in successive tests to compare the contribution of catchment attribute groups related to climate, terrain, land cover, and soil.  

We test if transforming the target variable has an effect on the xgboost model. For transformations that do not change the rank of the target variable, i would not expect to see an effect, however the metric used as the objective function may be sensitive to the distribution of target variables, that is sensitive to outliers.  We test the predictability of the following:

Skewness / Kurtosis (classical) tell you about asymmetry and tail‐heaviness, but can be noisy when your sample has extremes, which is a common issue in hydrology.  L-moments are linear combinations of order‐statistics, and are considered more robust under heavy tails.


In [ ]:
import os, sys
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import xarray as xr
from scipy.stats import skew, kurtosis
from math import comb

from bokeh.plotting import figure, show
from bokeh.layouts import gridplot, row, column
from bokeh.models import HoverTool, ColumnDataSource, Whisker, Legend, LegendItem
from bokeh.transform import factor_cmap, factor_mark
from bokeh.palettes import Category10, Category20, Category20c, Viridis
from bokeh.io import output_notebook
# from bokeh.palettes import Sunset10, Vibrant7
from pathlib import Path

# Add repo root to path
repo_root = Path(os.getcwd()).parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import Config

import xgboost as xgb

import utils.data_processing_functions as dpf
from utils.plotting import apply_tufte_style, ACADEMIC_FONT

from scipy.stats import linregress
output_notebook()

BASE_DIR = os.getcwd()

In [ ]:
# There is a ragged list construction warning in the xgb.train()
# # function because of the early stopping making eval_lists different lengths
# import warnings
# warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)


## Load Input Data

### Excluded due to no complete years of data >= 90% complete

* Genessee Creek at the Mouth - 08FA009
* McNair Creek near Port Mellon - 08GA037
* Canoe River near Valemount - 08NC003
* Big Quilcene River Near Quilcene, WA - 12052500
* Morey Creek above McChord Afb near Parkland, WA - 12090480
* North Fork Newaukum Creek Near Enumclaw, WA - 12107950
* Newaukum Creek Tributary Near Blacik Diamond, WA - 12108450
* May Creek near Issaquah, WA - 12119300
* Honey Creek near Renton, WA - 12119450
* Carpenter Creek near Bacon Rod near Mount Vernon, WA - 12200684
* Unnamed Tributary Massacre Bay on Orcas Island, WA - 12200762
* Whatcom Creek near Bellingham, WA - 12203000
* Hall Creek at Inchelium, WA - 12409500
* Dayebas Creek Near Haines, AK - 15056070
* Bonne Creek near Klawock, AK - 15081510

In [ ]:
# load the catchment characteristics
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'
attr_gdf = gpd.read_file(DATA_DIR / 'stn_attributes_with_5_spatial_partitions.geojson')
# # map the cluster numbers to the attributes dataframe
cluster_dict = attr_gdf.set_index('official_id')['5_spatial'].to_dict()
# map the climate column labels to match the special-symbol free versions
attr_gdf = attr_gdf.rename(Config.CLIMATE_COLUMN_MAPPER, axis=1)




In [ ]:
# load the FDC prediction results to get the processed stations
knn_results_fname = 'knn_all_results_formatted_kde.csv'
knn_results = pd.read_csv(DATA_DIR / 'fdc_estimation_results' / knn_results_fname)
processed_stations = knn_results['Official_ID'].unique()

## Define attribute groups

In [ ]:
terrain = ['drainage_area_km2', 'elevation_m', 'slope_deg', 'aspect_deg']
land_cover = [
    'land_use_forest_frac_2010', 'land_use_grass_frac_2010', 'land_use_wetland_frac_2010', 'land_use_water_frac_2010', 
    'land_use_urban_frac_2010', 'land_use_shrubs_frac_2010', 'land_use_crops_frac_2010', 'land_use_snow_ice_frac_2010']
climate = ['prcp', 'srad', 'swe', 'tmax', 'tmin', 'vp', 'high_prcp_freq', 'high_prcp_duration', 'low_prcp_freq', 'low_prcp_duration']
soil = ['logk_ice_x100', 'porosity_x100']

all_attributes = terrain + land_cover + soil + climate
len(all_attributes)
assert np.all([attr in attr_gdf.columns for attr in all_attributes]), f'Not all attributes are present in the attributes dataframe. Missing: {[attr for attr in all_attributes if attr not in attr_gdf.columns]}'

## Set Attribute Groupings

In [ ]:
all_test_results = {}
attribute_set_dict = {
    'climate': climate, 
    '+land_cover': land_cover,
    '+terrain': terrain, 
    '+soil': soil,
    'proximity': ['centroid_distance_km'],
    'physical_attributes': all_attributes,
    'proximity_plus_attributes': ['centroid_distance_km'] + all_attributes,
}

## Run XGBoost Regression

The same problem setup applies for the regression prediction problem which is to optimize the discriminant function and the input signal quantization simultaneously to minimize the error in predicting the KL divergence from catchment attributes.

Instead of predicting a scalar measure which is a feature of a single location, the key difference in this step is the target variable describes a measure of the difference in runoff between **pairs of locations**. This approach asks whether the **Kullback-Leibler Divergence** (KLD) of the distribution of unit area runoff between two locations can be predicted from the attributes of both catchments (and their differences) using the gradient boosted decision tree method, which is also capable of predicting continuous variables, in this case $D_\text{KL}$.

### Set trial parameters

In [ ]:
# define the amount of data to set aside for final testing
random_seed = 42
nfolds = 5
n_boost_rounds = 2500
n_optimization_rounds = 10
loss = 'reg:squarederror'
loss = 'reg:absoluteerror'
target_column = ['kld']
randomized_features = False

# cross validation parameters
optimize_cv_folds = False
cv_fold_seed = 83561
# limit the maximum distance to make the network 
# graph of station pairs more separable
max_centroid_distance = 500

attribute_sets = ['proximity', 'physical_attributes', 'proximity_plus_attributes']

### Train-test split

The input dataset is pairwise comparisons of just over 1300 (streamflow) monitored catchments, their attributes, and the attribute differences. After filtering for data concurrency (minimum 10 years, < 5 days missing per month) and maximum distance between basin centroids (500 km) we are left with roughly 225K pairs. The pairwise setup means that station data appears in more than one row. As a result, the attributes of stations can end up in both training and test sets if we simply split by randomly assigning rows to training or test sets. We can’t simply cut edges until the graph is separated because it is a generalization of the [“keeping a subset of vertices” problem in graph theory](https://en.wikipedia.org/wiki/Independent_set_(graph_theory)) which is NP-hard.

To address this issue we split the dataset spatially to create partitioned datasets for each fold and draw samples from within the “cluster” while filtering all edges between the fold and the rest of the set. The end goal is to generate training folds where the `official_id` does not appear in both training and test set, in either donor or target column. One problem remains, and that is to generate training and test sets with some assurance that the target variable distributions match to some degree, but this is less critical than ensuring there is no data leakage between training and test data.

Next we create training/test splits and visualize how the target variable distributions compare.

In [ ]:
# create the fold dictionary to initialize the train/test split for each fold
def create_fold_dict(df, gdf):
    cluster_ids = [int(e) for e in sorted(list(set(gdf['5_spatial'].values)))]
    print(f'The fold ids are: {cluster_ids}')
    fold_dict = {}
    for c in cluster_ids:
        cluster_stns = gdf.loc[gdf['5_spatial'] == c, 'official_id'].values
        # in-group edges
        dkl_sample_AND = df[(df['donor'].isin(cluster_stns)) & (df['target'].isin(cluster_stns))].copy()
        # out-of-group edges
        dkl_sample_NOR = df[(~df['donor'].isin(cluster_stns)) & (~df['target'].isin(cluster_stns))].copy()
        # assert that these are mutually exclusive groups
        and_official_ids = set(dkl_sample_AND['donor'].values + dkl_sample_AND['target'].values)
        nor_official_ids = set(dkl_sample_NOR['donor'].values + dkl_sample_NOR['target'].values)
        assert len(list(set(np.intersect1d(and_official_ids, nor_official_ids)))) == 0, 'stations in list are not unique'
        fold_dict[c] = {
            'test': dkl_sample_AND.index.values,
            'train': dkl_sample_NOR.index.values,
        }
    return fold_dict 

In [ ]:
def format_features(input_attributes):
    features = []
    for a in input_attributes:
        features.append(f"donor_{a}".lower())
        features.append(f"target_{a}".lower())
    return features

def add_attributes(attr_df, df, attribute_cols):
    """
    Adds attributes from the df_attributes to the df_relations based on the 'donor' and 'target' columns
    using map for efficient lookups.

    Parameters:
    df_attributes (pd.DataFrame): DataFrame with 'id' and attribute columns.
    df_relations (pd.DataFrame): DataFrame with 'donor' and 'target' columns.
    attribute_cols (list of str): List of attribute columns to add to df_relations.

    Returns:
    pd.DataFrame: Updated df_relations with added attribute columns.
    """
    # Create dictionaries for each attribute for quick lookup
    attr_dicts = {col: attr_df.set_index('official_id')[col].to_dict() for col in attribute_cols}

    # Add target attributes
    for col in attribute_cols:
        df[f'target_{col}'] = df['target'].map(attr_dicts[col])

    # Add donor attributes
    for col in attribute_cols:
        df[f'donor_{col}'] = df['donor'].map(attr_dicts[col])
    # for col in attribute_cols:
    #     df[f'{col}_diff'] = df[f'target_{col}'] - df[f'donor_{col}'] 

    return df

In [ ]:
def compute_empirical_cdf(data):
    sorted_data = np.sort(data)
    cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
    return sorted_data, cdf


def train_xgb_model(
    input_data, fold_no, cv_data, attributes, target, params, num_boost_rounds, log_transform_target=False
):    
    test_idxs = cv_data['test']
    train_idxs = cv_data['train']   

    test_data = input_data.iloc[test_idxs, :].copy()
    train_data = input_data.iloc[train_idxs, :].copy()

    X_train = train_data[attributes].values
    X_test = test_data[attributes].values
    # print(f'     Fold {fold_no}: {len(train_data)} training samples, {len(test_data)} test samples')
    
    if log_transform_target:
        Y_train = np.log(train_data[target].values)
        Y_test = np.log(test_data[target].values)
    else:
        Y_train = train_data[target].values
        Y_test = test_data[target].values

    dtrain = xgb.DMatrix(X_train, label=Y_train)
    dtest = xgb.DMatrix(X_test, label=Y_test)

    eval_list = [(dtrain, "train"), (dtest, "eval")]
    evals_result = {}
    bst = xgb.train(
        params,
        dtrain,
        num_boost_rounds,
        evals=eval_list,
        evals_result=evals_result,
        verbose_eval=0,
    )
    eval_keys = list(evals_result['train'].keys())
    if len(eval_keys) > 1:
        print(f' setting eval key to {eval_keys[0]} from {eval_keys}')

    eval_key = eval_keys[0]
    # Convert the lists to NumPy arrays with dtype=object
    train_perf = np.array(evals_result['train'][eval_key], dtype=object)
    test_perf = np.array(evals_result['eval'][eval_key], dtype=object)
    fold_learning = pd.DataFrame({
        'train': evals_result['train'][eval_key],
        'test': evals_result['eval'][eval_key],
        'fold': [fold_no]*len(evals_result['train'][eval_key]),
    })
    predicted = bst.predict(dtest)

    return train_perf, test_perf, fold_learning, predicted, Y_test, bst

In [ ]:
def run_xgb_trials_custom_CV(
    set_name,
    attributes,
    target,
    input_data,
    fold_dict, 
    n_optimization_rounds,
    num_boost_rounds,
    loss='reg:squarederror',
    random_seed=42,
    log_transform_target=False,
):
    """
    Custom CV refers to cross validation.  Custom cross validation means the 
    held-out set must be determined in a more robust way to avoid "data leakage".
    That is, the pairs making up the training, validation, and test sets must 
    be made up of pairings from unique sets of stations.
    """
    # select random hyperparameters for n_optimization_rounds
    sample_choices = np.arange(0.5, 1.0, 0.02)  # subsample and colsample percentages
    lr_choices = np.arange(0.001, 0.01, 0.001)  # learning rates
    etas = np.random.choice(lr_choices, n_optimization_rounds)
    subsamples = np.random.choice(sample_choices, n_optimization_rounds)
    colsamples = np.random.choice(sample_choices, n_optimization_rounds)
    num_boost_rounds = num_boost_rounds
    eval_key = loss.split(':')[1]

    # Format attributes: centroid_distance_km stays as-is (unprefixed column);
    # all others need donor_/target_ prefixes.
    has_distance = 'centroid_distance_km' in attributes
    attributes = format_features([a for a in attributes if a != 'centroid_distance_km'])
    if has_distance:
        attributes += ['centroid_distance_km']

    all_results = []
    all_trial_predictions = []
    output_target_cdfs = {}
    learning_rates = {}
    best_trial_mean = float("inf")
    for trial in range(n_optimization_rounds):
        lr, ss, cs = etas[trial], subsamples[trial], colsamples[trial]
        params = {
            "objective": loss,
            "eta": lr,
            "subsample": ss,  #  random subsampling of rows to prevent overfitting
            "colsample_bytree": cs, # random subsampling of columns to prevent overfitting
            "seed": random_seed,
            "device": "cuda",  # CUDA GPU
            "sampling_method": "gradient_based",
            "tree_method": "hist",
        }

        # k-fold cross validation       
        all_fold_results = []
        learning_arrays = []
        train_rounds = []
        cv_fold_perfs = []
        target_cdfs = []
        bst_models = {}
        for fold_no, cv_data in fold_dict.items():
            train_perf, test_perf, fold_learning, predicted, Y_test, bst = train_xgb_model(
                input_data,
                fold_no,
                cv_data,
                attributes,
                target,
                params,
                num_boost_rounds,
                log_transform_target=log_transform_target
            )
            learning_arrays.append(fold_learning)
            train_rounds.append(train_perf)

            bst_fname = f'{set_name}_trial{trial}_fold{fold_no}_model.json'
            bst_fpath = RESULTS_DIR / 'xgb_models' / bst_fname
            bst.save_model(bst_fpath)
            bst_models[f'fold_{fold_no}'] = bst_fpath

            # store the empirical cdf of the target values 
            # for this fold to compare in the learning rate plot
            ordered_data, fold_cdf = compute_empirical_cdf(Y_test)
            target_cdfs += [(ordered_data, fold_cdf)]

            test_ids = input_data.loc[cv_data['test'], ['donor', 'target']].values
            test_results = pd.DataFrame(
                {
                    "predicted": predicted,
                    "actual": Y_test,
                    "donor": [e[0] for e in test_ids],
                    "target": [e[1] for e in test_ids],
                    "trial": trial,
                    "fold": fold_no,
                }
            )

            # Store the metrics at the best round (minimum risk)
            # only used to show the minimum risk point in the learning rate plot
            cv_fold_perfs.append(test_perf[-1])
            all_fold_results.append(test_results)
            fold_no += 1

        all_test_predictions_df = pd.concat(all_fold_results)
        all_trial_predictions.append(all_test_predictions_df)

        # save the target cdfs for this trial for later selection
        output_target_cdfs[trial] = target_cdfs
        # learning curves
        convergence_df = pd.concat(learning_arrays)
        learning_rates[trial] = convergence_df

        # mean t
        trial_mean_test_perf = np.mean(cv_fold_perfs)
        results_dict = {
            'trial': trial,
            f'test_{eval_key}_mean': trial_mean_test_perf,
            f'test_{eval_key}_median': np.median(cv_fold_perfs),
            f'test_{eval_key}_stdev': np.std(cv_fold_perfs),
        }        
        results_dict.update(params)
        all_results.append(results_dict)
            
        if round(trial_mean_test_perf, 2) < round(best_trial_mean, 2):
            best_trial_mean = trial_mean_test_perf
            print(f'    New best result: {eval_key}={best_trial_mean:.2f} (trial {trial})')

        if (trial > 0) & (trial % 10 == 0):
            print(f"   completed {trial}/{n_optimization_rounds}")
    
    # save the best trial results
    # best_trial_test_predictions.to_csv(results_fpath)
    all_trial_predictions_df = pd.concat(all_trial_predictions, ignore_index=True)
    results_all_trials = pd.DataFrame(all_results)

    # get the mean and standard deviation of the error metrics over all trials
    all_trials_mean = results_all_trials[f"test_{eval_key}_mean"].mean()
    all_trials_stdev = results_all_trials[f"test_{eval_key}_mean"].std()
    print(
        f"    {all_trials_mean:.2f} ± {2*all_trials_stdev:.2f} mean {eval_key} "
        f"    (of {len(results_all_trials)} hyperparameter optimization rounds.)"
    )
    output = {
        'all_trial_predictions_df': all_trial_predictions_df,
        'results_all_trials': results_all_trials,
        'output_target_cdfs': output_target_cdfs,
        'learning_rates': learning_rates,
    }
    return output


In [ ]:
def predict_KLD_from_attributes(bits, df, target_variable, all_stations,
                                attribute_sets, loss_function=None, n_boost_rounds=100, random_seed=42, 
                              n_cv_fold_optimization_trials=20, log_transform_target=False):

    all_results = {}
    gdf = attr_gdf[attr_gdf['official_id'].isin(all_stations)].copy()
    fold_dict = create_fold_dict(df, gdf)

    # evaluate each feature set independently
    for attribute_set in attribute_sets:
        print(f'  Processing {attribute_set} attribute set: {target_variable}')
        # initialize the predictor variables (features)
        features = attribute_set_dict[attribute_set] 
        trial_result = run_xgb_trials_custom_CV(
                attribute_set, features, target_variable, df, fold_dict, 
                n_cv_fold_optimization_trials, n_boost_rounds, 
                loss=loss_function, random_seed=random_seed,
                log_transform_target=log_transform_target,
        )
        # store the test set predictions and actuals
        all_results[attribute_set] = trial_result
    return all_results

In [ ]:
rev_date = '20251214'
target_col = f'kld'
log_transform_target = True
estimator_type = 'kde'
eval_key = loss.split(':')[1]

results_folder = os.path.join(BASE_DIR, 'results', 'KLD_prediction_results')
if not os.path.exists(results_folder):
    os.makedirs(results_folder)

for bits in [8]:#[6, 8, 10, 12]:
    kld_pairs_batch_fpath = Path(BASE_DIR) / 'data'/ 'kld_batches' / f'{bits}_bits' / f'pairwise_kld_{bits}_bits_{estimator_type}.csv'
    
    test_results_fname = f'{target_col}_results_{eval_key}_{estimator_type}_{bits}bits_{rev_date}.npy'
    if randomized_features:
        test_results_fname = f'{target_col}_results_{eval_key}_{estimator_type}_{bits}bits_{rev_date}_random.npy'
    test_results_fpath = os.path.join(results_folder, test_results_fname)
    if os.path.exists(test_results_fpath):
        print('processed and loading: ', test_results_fname)
        all_test_results = np.load(test_results_fpath, allow_pickle=True).item()
    else:
        
        kl_df = pd.read_csv(kld_pairs_batch_fpath, dtype={'donor': str, 'target': str})
        print(f'  {len(kl_df)} pairs in the training sample')        
        all_donors = set(kl_df['donor'].values)
        all_targets = set(kl_df['target'].values)
        all_stations = set(all_donors.union(all_targets))
        # include only donor and target stations that are in processed_stations
        kl_df = kl_df[(kl_df['donor'].isin(processed_stations)) & (kl_df['target'].isin(processed_stations))].copy()
        # reset the index so that the train/test splits work correctly in the CV function
        kl_df.reset_index(inplace=True, drop=True)

        # add the attributes into the input dataset
        input_df = add_attributes(attr_gdf, kl_df, all_attributes)
        if randomized_features:
            for attr in all_attributes:
                input_df[f'target_{attr}'] = np.random.permutation(input_df[f'target_{attr}'].values)
                input_df[f'donor_{attr}'] = np.random.permutation(input_df[f'donor_{attr}'].values)
        all_test_results = predict_KLD_from_attributes(
            bits, input_df, target_col, all_stations, attribute_sets,
            loss_function=loss, n_boost_rounds=n_boost_rounds, random_seed=random_seed,
            n_cv_fold_optimization_trials=n_optimization_rounds,
            log_transform_target=log_transform_target
        )
        np.save(test_results_fpath, all_test_results)

In [ ]:
def load_result(rev_date, estimator_type, eval_key='absoluteerror', bitrate=6, randomized_features=False):

    fpath = RESULTS_DIR / 'KLD_prediction_results' / f'kld_results_{eval_key}_{estimator_type}_{bitrate}bits_{rev_date}.npy'
    if randomized_features:
        fpath = RESULTS_DIR / 'KLD_prediction_results' / f'kld_results_{eval_key}_{estimator_type}_{bitrate}bits_{rev_date}_random.npy'
    return np.load(fpath, allow_pickle=True).item()

In [ ]:
from bokeh.transform import factor_cmap, linear_cmap
from bokeh.models import ColumnDataSource, LinearAxis, Range1d, CustomJSTickFormatter, FixedTicker
from bokeh.io import output_notebook

from bokeh.palettes import Sunset10, Vibrant7, Category20, Bokeh6, Bokeh7, Bokeh8, Greys256
from utils.plotting import apply_tufte_style

def compute_uncertainty_bands(log10_xx, percent_residual):

    df_for_bins = pd.DataFrame({'actual_log10': log10_xx, 'percent_residual': percent_residual}).dropna()
    
    bin_assignments = pd.qcut(
        df_for_bins['actual_log10'], 
        q=min(50, df_for_bins.shape[0]), 
        labels=False, 
        duplicates='drop'
        )

    df_for_bins = df_for_bins.assign(bin=bin_assignments)
    df_for_bins = df_for_bins.dropna(subset=['bin'])
    df_for_bins['bin'] = df_for_bins['bin'].astype(int)
    bin_rows = []
    for _, grp in df_for_bins.groupby('bin'):
        if grp.empty:
            continue
        x_mid = float(np.median(grp['actual_log10']))
        pct_vals = np.percentile(grp['percent_residual'], [2.5, 50, 97.5])
        delta = 1.0 + pct_vals / 100.0
        valid = delta > 0
        delta = np.where(valid, delta, np.nan)
        with np.errstate(divide='ignore', invalid='ignore'):
            log_offsets = np.log10(delta)
        bin_rows.append({
            'x_log10': x_mid,
            'y_p2.5_log10': x_mid + log_offsets[0],
            'y_p50_log10': x_mid + log_offsets[1],
            'y_p97.5_log10': x_mid + log_offsets[2],
            'count': len(grp)
        })

    band_df = pd.DataFrame(bin_rows)
    band_df = band_df.dropna(subset=['x_log10'])
    band_df = band_df.dropna(how='all', subset=['y_p2.5_log10', 'y_p50_log10', 'y_p97.5_log10'])
    band_df = band_df.sort_values('x_log10')
            
    return band_df

In [ ]:
# layout_dict = {}
# reg_plots_dict = {}
res_dict = {}
uncertainty_bands = {}
plots = []
estimator_type = 'kde'
# estimator_type = 'obs'
first_scatter = None
err_models = {}
al_size = '18pt'
ml_size = '16pt'
ll_size = '14pt'

# key for the full-attribute set in attribute_set_dict
FULL_SET_KEY = 'physical_attributes'#,'proximity_plus_attributes'

def apply_axis_fonts(fig):
    for axis in list(fig.xaxis) + list(fig.yaxis):
        axis.axis_label_text_font_size = al_size
        axis.axis_label_text_font = ACADEMIC_FONT
        axis.major_label_text_font_size = ml_size
        axis.major_label_text_font = ACADEMIC_FONT


def apply_legend_fonts(fig):
    fig.legend.label_text_font_size = ll_size
    fig.legend.label_text_font = ACADEMIC_FONT


for br in [8]:
    all_results = load_result(rev_date, estimator_type, bitrate=br, eval_key=eval_key, randomized_features=randomized_features)
    test_rmse, test_mae = [], []
    y, y2, lbs, ubs = [], [], [], []
    err_models[br] = {}
    metric = f'test_{eval_key}_mean'
    for e in all_results.keys():
        # test_data_df = all_results[e]['all_results']
        test_data_df = all_results[e]['results_all_trials']
        
        # sort by the trial mean test performance
        test_data_df = test_data_df.sort_values(metric)
        test_data_df.reset_index(inplace=True, drop=True)
        # get the len(test_data_df) // 2 th trial (the median trial)
        median_trial = test_data_df.loc[len(test_data_df) // 2, 'trial']
        test_scores = test_data_df[metric].values
        lb, ub = np.min(test_scores), np.max(test_scores)
        if 'absolute' in metric:
            all_trial_preds = all_results[e]['all_trial_predictions_df']
            trial_mape = (
                all_trial_preds
                .assign(pct_err=100 * np.expm1(
                    np.abs(all_trial_preds['predicted'] - all_trial_preds['actual'])
                ))
                .groupby('trial')['pct_err']
                .mean()
            )
            # selected_preds = all_trial_preds[all_trial_preds['trial'] == median_trial]
            # med_mape = 100 * np.expm1(
            #     np.abs(selected_preds['predicted'] - selected_preds['actual'])
            # ).mean()
            med_mape = trial_mape.median()
            print(f'{metric} {e} mean absolute error: {med_mape:.2f}%')
            y2.append(med_mape)
            lb, ub = trial_mape.min(), trial_mape.max()

        else:
            y2.append(test_data_df[metric].median())
            lb, ub = test_data_df[metric].min(), test_data_df[metric].max()
            # lb, ub left as raw MSE values
        lbs.append(lb)
        ubs.append(ub)

    label_map = {
        'proximity': 'Proximity',
        'physical_attributes': 'Physical',
        'proximity_plus_attributes': 'Proximity + Phys.',
    }
    display_sets = [label_map.get(s, s) for s in attribute_sets]
    source = ColumnDataSource({'x': display_sets, 'y2': y2, 'lb': lbs, 'ub': ubs})
    if 'squared' in metric:
        units = 'KLD^2' if not log_transform_target else 'log-KLD^2'
        ylab = f'{units}'
    elif 'absolute' in metric:
        units = 'KLD' if not log_transform_target else 'log-KLD'
        ylab = f'100 \u00d7 (exp|e| - 1) (%)'
        ylab = 'MAPE (%)'
        # ylab = 'MAE (log KLD)'
        # ylab = f'Absolute multiplicative error (%)'
    if len(plots) == 0:
        fig = figure(x_range=display_sets, toolbar_location='above', y_range=(0.0, max(y2) * 1.1))
    else:
        fig = figure(x_range=display_sets, y_range=plots[0].y_range, toolbar_location='above')
    # fig.line('x', 'y1', legend_label='rmse', color='green', source=source, line_width=3)
    fig.scatter('x', 'y2', legend_label=f'Median ({n_optimization_rounds} trials)', 
                color='dodgerblue', source=source, size=10)
    # add a whisker
    w = Whisker(source=source, base='x', upper='ub', lower='lb', line_color='black', line_width=2)
    fig.add_layout(w)

    fig.legend.background_fill_alpha = 0.6
    fig.yaxis.axis_label = ylab
    fig.xaxis.axis_label = 'Feature group'
    # apply_tufte_style(fig)
    fig.legend.location = 'bottom_left'
    apply_axis_fonts(fig)
    apply_legend_fonts(fig)

    plots.append(fig)
    
    all_trial_results = all_results[FULL_SET_KEY]['all_trial_predictions_df'].copy()
    # recompute median_trial for the full set
    full_set_trial_df = all_results[FULL_SET_KEY]['results_all_trials'].sort_values(metric)
    full_set_trial_df.reset_index(inplace=True, drop=True)
    median_trial = full_set_trial_df.loc[len(full_set_trial_df) // 2, 'trial']
    test_data_df = full_set_trial_df
    median_trial_df = all_trial_results[all_trial_results['trial'] == median_trial].copy()
    
    # best_result['predicted'] = best_result['predicted'].clip(lower=1e-5)
    # assertno nans
    assert np.sum(np.isnan(median_trial_df['predicted'])) == 0, 'nans in predicted values'
    assert np.sum(np.isnan(median_trial_df['actual'])) == 0, 'nans in actual values'

    xx, yy = median_trial_df['actual'], median_trial_df['predicted']
    # xx and yy are in log space, so the residual is the 
    # log of the ratio of predicted to actual
    percent_residual = np.expm1(yy - xx) * 100.0

    res_dict[br] = pd.DataFrame({'actual': xx, 'predicted': yy, 'residual': yy - xx, 'percent_residual': percent_residual})

    assert np.sum(np.isnan(xx)) == 0, 'nans in actual values'
    assert np.sum(np.isnan(yy)) == 0, 'nans in predicted values'
    xmin, ymin = np.nanmin(xx), np.nanmin(yy)
    xmax, ymax = np.nanmax(xx), np.nanmax(yy)

    slope, intercept, r, p, se = linregress(xx, yy)
    print(f'   R²: {r**2:.3f}, slope: {slope:.3f}, intercept: {intercept:.3f}')
    # sfig = figure(title=f'Test: {b} bits best model {best_rmse_set} (N={len(best_result)})', toolbar_location='above')
    if first_scatter is None:
        sfig = figure(title=f'', toolbar_location='above', x_range=(-1.8, 1.8), y_range=(-1.8, 1.8))
    else:
        sfig = figure(title=f'', toolbar_location='above', x_range=first_scatter.x_range, y_range=first_scatter.y_range)

    # Create hexbin plot
    # binsize=0.1 if bits <=10 else 0.05
    xx_exp, yy_exp = np.exp(xx), np.exp(yy)
    log10_xx, log10_yy = np.log10(xx_exp), np.log10(yy_exp)
    
    band_df = compute_uncertainty_bands(log10_xx, percent_residual)
    check = band_df[band_df['x_log10'] < 0][['x_log10', 'y_p2.5_log10']]
    print(check.assign(lower_above_1to1 = check['y_p2.5_log10'] > check['x_log10']))
    uncertainty_bands[br] = band_df

    binsize = 0.05
    hex_renderer, hex_data = sfig.hexbin(log10_xx, log10_yy, size=binsize, hover_color='pink', hover_alpha=0.8)

    # Add color mapping based on bin counts
    counts = hex_data['counts']  # Extract the counts from the source
    mapper = linear_cmap(field_name='counts', palette=Greys256[::-1], low=min(counts), high=max(counts))
    log_formatter = CustomJSTickFormatter(code="""
        const superscripts = {'0':'⁰','1':'¹','2':'²','3':'³','4':'⁴','5':'⁵','6':'⁶','7':'⁷','8':'⁸','9':'⁹','-':'⁻'};
        const rounded = Math.round(tick);

        if (Math.abs(tick - rounded) > 1e-6) {
            return '';
        }

        return '10' + String(rounded).split('').map(c => superscripts[c] || c).join('');
    """)
    integer_ticker = FixedTicker(ticks=list(range(-4, 5)))
    sfig.xaxis.formatter = log_formatter
    sfig.xaxis.ticker = integer_ticker
    sfig.yaxis.formatter = log_formatter
    sfig.yaxis.ticker = FixedTicker(ticks=list(range(-4, 5)))

    # Plot the hex tiles using the color mapping
    sfig.hex_tile(q='q', r='r', size=binsize, line_color=None, source=hex_data, fill_color=mapper)

    band_df = uncertainty_bands.get(br)
    if band_df is not None and not band_df.empty:
        sfig.line(band_df['x_log10'].tolist(), band_df['y_p50_log10'].tolist(), 
                  color='salmon', line_dash='dashed', line_width=1.5, legend_label='Median')
        sfig.line(band_df['x_log10'].tolist(), band_df['y_p97.5_log10'].tolist(), 
                  color='firebrick', line_dash='dotted', line_width=3, legend_label='95% PI')
        sfig.line(band_df['x_log10'].tolist(), band_df['y_p2.5_log10'].tolist(), 
                  color='firebrick', line_dash='dotted', line_width=3, legend_label='95% PI')
        # Express lower/upper bounds as fractions of predicted value
        predicted_vals = np.power(10, band_df['y_p50_log10'])
        lb_vals = (np.power(10, band_df['x_log10'] - band_df['y_p2.5_log10']) / predicted_vals).tolist()
        ub_vals = (np.power(10, band_df['y_p97.5_log10'] - band_df['x_log10']) / predicted_vals).tolist()
        err_models[br] = {
            'predicted': predicted_vals.tolist(),
            '2.5_pct': lb_vals,
            '97.5_pct': ub_vals,
        }
        # foo = pd.DataFrame(err_models[br])
        # print(foo)
        # print(asdf)

    xpred = np.linspace(min(xx), max(xx), 100)
    ybf = [slope * e + intercept for e in xpred]
    # sfig.line(xpred, ybf, color='red', line_width=2,
    #           alpha=0.7, line_dash='dashed', legend_label=f'R²={r**2:.2f}')
    # plot a 1:1 line
    sfig.line([-2, 2], [-2, 2], color='grey', line_dash='dotted', line_width=3, legend_label='1:1')
    sfig.xaxis.axis_label = 'Actual KLD [bits/sample]'
    sfig.yaxis.axis_label = 'Predicted KLD [bits/sample]'
    sfig.legend.background_fill_alpha = 0.4
    sfig.legend.location = 'top_left'
    if first_scatter is None:
        first_scatter = sfig
    # apply_tufte_style(sfig)
    apply_axis_fonts(sfig)
    apply_legend_fonts(sfig)

    plots.append(sfig)

    # plot the test set convergence for the median trial
    ymin = min(100, 100*np.expm1(0.8 * min(test_data_df[metric].values)))
    ymax = 1.1 * 100*np.expm1(max(test_data_df[metric].values))
    cfig = figure(x_axis_type='log')#, y_range=(ymin, ymax), toolbar_location='above')

    convergence_df = all_results[FULL_SET_KEY]['learning_rates'][median_trial].copy()

    # Pivot the data to get separate columns for each fold
    train_pivot = convergence_df.pivot(columns='fold', values='train')
    test_pivot = convergence_df.pivot(columns='fold', values='test')

    # Rename the columns to indicate folds
    train_pivot.columns = [f'fold_{col}' for col in train_pivot.columns]
    test_pivot.columns = [f'fold_{col}' for col in test_pivot.columns]
    train_pivot['mean'] = train_pivot.mean(axis=1)
    test_pivot['mean'] = test_pivot.mean(axis=1)
    fold_nos = sorted(list(set(convergence_df['fold'])))

    for fn in fold_nos:
        cfig.line(test_pivot.index, 100*np.expm1(test_pivot[f'fold_{fn}']), legend_label='Test folds',
                  line_alpha=0.75, line_width=1.1, line_color='red')
        cfig.line(train_pivot.index, 100*np.expm1(train_pivot[f'fold_{fn}']), legend_label='Train fold', 
                  line_alpha=0.75, line_width=1.1, line_color='grey')

    # find the minimum predictive risk
    min_pred_risk_idx = test_pivot['mean'].idxmin()
    if min_pred_risk_idx == max(test_pivot['mean'].index):
        print(f'Min prediction risk occurs at the maximum iteration, try increasing the number of boosting rounds')

    min_pred_risk_y1 = train_pivot.loc[min_pred_risk_idx, :].min()
    min_pred_risk_y2 = test_pivot.loc[min_pred_risk_idx, :].max()
    pr_min, pr_max = 100*np.expm1(min_pred_risk_y1), 100*np.expm1(min_pred_risk_y2)
    cfig.line([min_pred_risk_idx, min_pred_risk_idx], [pr_min, pr_max], legend_label='Min risk', color='green', line_width=2, line_dash='dashed')

    cfig.xaxis.axis_label = 'Iteration'
    cfig.yaxis.axis_label = 'MAE (log KLD)'
    cfig.legend.background_fill_alpha = 0.5
    cfig.legend.location = 'bottom_left'
    apply_axis_fonts(cfig)
    apply_legend_fonts(cfig)

    plots.append(cfig)

    # plot the cdfs of the target variables in each fold to compare
    cdffig = figure(x_range=sfig.x_range)
    
    
    cdf_arrays = all_results[FULL_SET_KEY]['output_target_cdfs'][median_trial]
    cdf_pcts = np.linspace(0.001, 100.0, 500)
    for (cdfx, cdfy) in cdf_arrays:
        cdf_x = np.percentile(cdfx, cdf_pcts)
        cdf_df = pd.DataFrame({'x': cdf_x, 'y': cdf_pcts})
        cdffig.line(cdf_x *np.log10(np.e), cdf_pcts, color='grey', line_alpha=0.6, line_width=2, legend_label='Fold CDFs')
    # plot the overall cdf of the target variable across all folds
    all_cdf_x = np.percentile(np.concatenate([cdfx for (cdfx, cdfy) in cdf_arrays]), cdf_pcts)
    dkl_pcts = np.percentile(np.exp(all_cdf_x), (25, 50, 75))
    print(f'   Overall KLD IQR:(25th pct: {dkl_pcts[0]:.2f}, 50th pct: {dkl_pcts[1]:.2f}, 75th pct: {dkl_pcts[2]:.2f})')
    cdffig.line(all_cdf_x*np.log10(np.e), cdf_pcts, color='black', line_width=3, line_dash='dashed', legend_label='Overall CDF')
    cdffig.xaxis.axis_label = 'Observed Values [bits/sample]' if log_transform_target else 'log Observed Values [bits/sample]'
    cdffig.yaxis.axis_label = 'Pr(x <= X)'
    cdffig.legend.location = 'top_left'
    # cdffig.xaxis.formatter = log_formatter
    cdffig.xaxis.formatter = log_formatter
    cdffig.xaxis.ticker = integer_ticker
    apply_axis_fonts(cdffig)
    apply_legend_fonts(cdffig)
    plots.append(cdffig)


### Interpreting percentile bands
The dashed overlays in the predicted-versus-observed panel summarise the adaptive log-binned residuals. The median line traces the systematic multiplicative bias, while the 10th and 90th percentile curves enclose an uncertainty fan that widens where the model exhibits larger percent errors. Because the envelopes are built on quantile bins, each band represents a comparable number of samples, so changes in spread with magnitude highlight genuine heteroscedastic structure rather than sampling density effects.

In [ ]:
from utils.plotting import save_figure

all_layout = gridplot(plots, ncols=2, width=460, height=340, toolbar_location=None)
# export to html file
if randomized_features:
    fname = f'kld_prediction_scatter_{estimator_type}_{eval_key}_{eval_key}_random'
else:
    fname = f'kld_prediction_scatter_{estimator_type}_{eval_key}'

# save_figure(all_layout, figure_name=fname, tufte=False)
show(all_layout)

In [ ]:
mape_proximity, mape_all = 234, 108
all_actual = all_results[FULL_SET_KEY]['all_trial_predictions_df']['actual']
median_kld_bits = np.exp(np.median(all_actual))  # convert from ln(KLD) to bits
delta_bits = (mape_proximity - mape_all) / 100 * median_kld_bits
frac_improvement  = 1 - (mape_all / mape_proximity)
print(f'MAPE improvement from proximity to all_descriptors: {frac_improvement:.2%}')

print(f'Median KLD: {median_kld_bits:.3f} bits, delta: {delta_bits:.3f} bits')

frac_lt_zero = (all_actual < 0).mean()
print(f'Fraction of actual KLD values < 0: {frac_lt_zero:.2%}')

diffs = median_trial_df['predicted'] - median_trial_df['actual']


In [ ]:
p = figure(#title='Residuals vs Actual', 
           toolbar_location='above', width=600, height=300)
hist, edges = np.histogram(diffs, bins=50)
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], fill_color='steelblue', 
       line_color='white', alpha=0.5)
p.xaxis.axis_label = 'Predicted - Actual (log KLD)'
p.yaxis.axis_label = 'Count'
apply_axis_fonts(p)
show(p)

In [ ]:
from scipy.stats import kurtosis, kurtosistest
print(f'Excess kurtosis: {kurtosis(diffs):.2f}')   # 0 = Normal, 3 = Laplace
k_stat, k_p = kurtosistest(diffs)
print(f'Kurtosis test p-value: {k_p:.4f}')

In [ ]:

from scipy.stats import shapiro
tests = []
# run a suite of statistical test of normality to determine if the distribution 
# of residuals (predicted - actual) is approximately normal,
# which would justify using a t-test to compare the means of the two groups

for i in range(1000):
    sample_diffs = np.random.choice(diffs, size=5000, replace=True)
    t_stat, p_value = shapiro(sample_diffs)
    tests.append({'t_stat': t_stat, 'p_value': p_value})

test_df = pd.DataFrame(tests)
print(f'Shapiro-Wilk test for normality of residuals: mean t-statistic = {test_df["t_stat"].mean():.3f}, mean p-value = {test_df["p_value"].mean():.3f}')



In [ ]:

from shap import TreeExplainer
# compute the shapley treeexplainer
# Retrain on all data using median-trial hyperparameters
# (the current train_xgb_model doesn't return the booster, so retrain once)

median_trial_row = test_data_df.loc[len(test_data_df) // 2]
params = {
    'objective': loss,
    'eta': float(median_trial_row['eta']),
    'subsample': float(median_trial_row['subsample']),
    'colsample_bytree': float(median_trial_row['colsample_bytree']),
    'seed': 42,
    'device': 'cuda',
    'sampling_method': 'gradient_based',
    'tree_method': 'hist',
}
kl_df = pd.read_csv(kld_pairs_batch_fpath, dtype={'donor': str, 'target': str})
print(f'  {len(kl_df)} pairs in the training sample')        
all_donors = set(kl_df['donor'].values)
all_targets = set(kl_df['target'].values)
all_stations = set(all_donors.union(all_targets))
# include only donor and target stations that are in processed_stations
kl_df = kl_df[(kl_df['donor'].isin(processed_stations)) & (kl_df['target'].isin(processed_stations))].copy()
# reset the index so that the train/test splits work correctly in the CV function
kl_df.reset_index(inplace=True, drop=True)

# add the attributes into the input dataset
input_df = add_attributes(attr_gdf, kl_df, all_attributes)
feat_cols_all = format_features([a for a in all_attributes if a != 'centroid_distance_km']) + ['centroid_distance_km']
X_all = input_df[feat_cols_all].values
Y_all = np.log(input_df['kld'].values)
dtrain = xgb.DMatrix(X_all, label=Y_all)
bst = xgb.train(params, dtrain, n_boost_rounds, verbose_eval=False)

# SHAP values
import shap
explainer = shap.TreeExplainer(bst)
shap_vals = explainer.shap_values(X_all)  # (n_pairs, n_features)

# Collapse donor/target to base attribute
base_imp = {}
for i, col in enumerate(feat_cols_all):
    base = col.removeprefix('donor_').removeprefix('target_')
    base_imp[base] = base_imp.get(base, 0) + np.abs(shap_vals[:, i]).mean()

imp = pd.Series(base_imp).sort_values(ascending=False)

In [ ]:
feat_to_group = {}
for attr in terrain:     feat_to_group[attr] = 'terrain'
for attr in land_cover:  feat_to_group[attr] = 'land_cover'
for attr in climate:     feat_to_group[attr] = 'climate'
for attr in soil:        feat_to_group[attr] = 'soil'
feat_to_group['centroid_distance_km'] = 'proximity'

GROUP_COLORS = {
    'climate':    '#4c72b0',
    'terrain':    '#f28e2b',
    'land_cover': '#59a14f',
    'soil':       '#e15759',
    'proximity':  '#888888',
}

features_ordered = imp.index.tolist()
display_labels = [f.replace('_2010', '') for f in features_ordered]

colors = [GROUP_COLORS.get(feat_to_group.get(f, ''), '#aaaaaa') for f in features_ordered]

src = ColumnDataSource({
    'display': display_labels,
    'importance': imp.values.tolist(),
    'color': colors,
    'group': [feat_to_group.get(f, 'other') for f in features_ordered],
})

ph = max(300, len(features_ordered) * 22)
shap_fig = figure(
    y_range=display_labels[::-1],
    width=680, height=ph,
    toolbar_location=None,
    x_axis_label='Mean |SHAP| (summed donor + target)',
)
shap_fig.hbar(y='display', right='importance', height=0.7, color='color', source=src, alpha=0.85)

# Dummy off-screen renderers so Bokeh has a colour to draw in the legend swatch
legend_items = []
for grp, color in GROUP_COLORS.items():
    r = shap_fig.hbar(
        y=[-9999], right=[0], height=0.001,
        fill_color=color, line_color=None, fill_alpha=0.85,
    )
    legend_items.append(LegendItem(label=grp, renderers=[r]))

legend = Legend(items=legend_items, location='bottom_right')
legend.label_text_font = ACADEMIC_FONT
legend.label_text_font_size = '13pt'
shap_fig.add_layout(legend)

shap_fig.yaxis.major_label_text_font_size = '12pt'
shap_fig.yaxis.major_label_text_font = ACADEMIC_FONT
apply_axis_fonts(shap_fig)

save_figure(shap_fig, figure_name='shap_importance_kld', tufte=False)
show(shap_fig)

## Conditional summary: actual KLD given quantized predicted KLD

Median (dashed) and 90% CI of actual KLD within each adaptive predicted-KLD bin, overlaid on the hexbin density. Bins are at least 0.1 bits wide and merged at the tails until each holds at least 1 000 pairs.


In [ ]:
from bokeh.models import Band

br = list(res_dict.keys())[-1]
df_cd = res_dict[br].copy()
pred_bits   = np.exp(df_cd['predicted'].values)
actual_bits = np.exp(df_cd['actual'].values)

def make_pred_bins(values, min_width=0.1, min_count=1000):
    lo, hi = float(np.nanmin(values)), float(np.nanmax(values))
    raw = np.arange(np.floor(lo / min_width) * min_width,
                    np.ceil(hi  / min_width) * min_width + 1e-9, min_width)
    raw = np.unique(np.r_[lo, raw[(raw > lo) & (raw < hi)], hi])
    counts, _ = np.histogram(values, bins=raw)
    edges, acc = [raw[0]], 0
    for i, c in enumerate(counts):
        acc += c
        if acc >= min_count:
            edges.append(raw[i + 1])
            acc = 0
    if edges[-1] < hi:
        edges[-1] = hi
    edges = np.array(edges)
    counts2, _ = np.histogram(values, bins=edges)
    rev, acc = [edges[-1]], 0
    for i in range(len(counts2) - 1, -1, -1):
        acc += counts2[i]
        if acc >= min_count:
            rev.append(edges[i])
            acc = 0
    if rev[-1] > lo:
        rev[-1] = lo
    return np.array(sorted(rev))


x_bin_edges = make_pred_bins(pred_bits, min_width=0.2, min_count=1000)
bin_counts, _ = np.histogram(pred_bits, bins=x_bin_edges)
print(f'{len(x_bin_edges)-1} bins | widths (bits): '
      f'min={np.diff(x_bin_edges).min():.3f}, max={np.diff(x_bin_edges).max():.2f}')
print(f'samples per bin: min={bin_counts.min()}, max={bin_counts.max()}')

bin_idx = np.clip(np.digitize(pred_bits, x_bin_edges) - 1, 0, len(x_bin_edges) - 2)
rows = []
for ci in range(len(x_bin_edges) - 1):
    mask = bin_idx == ci
    if mask.sum() == 0:
        continue
    x_mid = np.sqrt(x_bin_edges[ci] * x_bin_edges[ci + 1])  # geometric mean
    p5, p50, p95 = np.percentile(actual_bits[mask], [5, 50, 95])
    rows.append({'x': x_mid, 'lo': p5, 'mid': p50, 'hi': p95})

cond_df = pd.DataFrame(rows).sort_values('x').reset_index(drop=True)
band_src = ColumnDataSource(cond_df)

# random subset of points for scatter
rng = np.random.default_rng(42)
print(len(pred_bits), len(actual_bits))
n_sample = min(10000, len(pred_bits))
idx = rng.choice(len(pred_bits), size=n_sample, replace=False)
scatter_src = ColumnDataSource({'x': pred_bits[idx], 'y': actual_bits[idx]})

xy_range = (0.1,
            float(max(pred_bits.max(), actual_bits.max())) * 1.1)

cfig2 = figure(
    width=580, height=400,
    # toolbar_location='above',
    toolbar_location=None,
    x_axis_type='log', y_axis_type='log',
    x_range=xy_range, y_range=xy_range,
    x_axis_label='Predicted KLD [bits/sample]',
    y_axis_label='Actual KLD [bits/sample]',
    output_backend='webgl',
)

cfig2.scatter('x', 'y', source=scatter_src,
              size=2, color='steelblue', alpha=0.45, line_color=None)

band = Band(base='x', lower='lo', upper='hi', source=band_src,
            fill_color='steelblue', fill_alpha=0.3, line_color=None)
cfig2.add_layout(band)

cfig2.line('x', 'mid', source=band_src,
           line_color='red', line_width=2.5, line_dash='dashed',
           legend_label='Conditional Median')
cfig2.quad(left=[-999], right=[-999], bottom=[0], top=[1],
           fill_color='steelblue', fill_alpha=0.3, line_color=None,
           legend_label='90% CI')

# 1:1 line
ref = [0.1, xy_range[1]]
cfig2.line(ref, ref, line_color='black', line_width=1.5,
           line_dash='dotted', legend_label='1:1')

cfig2.legend.location = 'top_left'
cfig2.legend.background_fill_alpha = 0.5
apply_axis_fonts(cfig2)
apply_legend_fonts(cfig2)
fname = f'kld_prediction_conditional_error_{estimator_type}_{eval_key}'
save_figure(cfig2, figure_name=fname, tufte=False)
show(cfig2)


## Discussion

- ...


## Citations

```{bibliography}
:filter: docname in docnames
```